In [ ]:
from astropy.time import Time
import astropy.units as u
import json
import pandas as pd

In [ ]:
# Function to read the JSON settings file
def get_config():
  
  # Get settings' directory
  settings_dir = r"../config/settings.json"

  # Open JSON file
  with open(settings_dir, "r") as file:
      settings = json.load(file)
  return settings

# Function to import local classification database
def get_localdatabase():
  
  # Data directory
  data_dir = r"../data/body_classification.csv"
  
  # Read CSV file
  return pd.read_csv(data_dir, delimiter=',')

In [55]:
settings = get_config()
asteroids = get_localdatabase()

obj_state = settings["body"]["fields"]['state']

In [53]:
obj_state

['provisional', 'numbered']

In [56]:
# Handle state
if None in obj_state or ('provisional' in obj_state and 'numbered' in obj_state):
  print('Both')
  bodies = asteroids
elif 'provisional' in obj_state:
  print('Provisional!')
  bodies = asteroids[asteroids.is_provisional]
elif 'numbered' in obj_state:
  print('Numbered!')
  bodies = asteroids[asteroids.is_numbered]

bodies = bodies.drop(columns=['is_provisional', 'is_numbered'])

Both


In [47]:
import re

for i in bodies['designation']:
  if re.match(r'\(\d', i):
    print('a')

In [ ]:
# Functions to standardize input data

# Helper function to search body names
def _search_bodies(fields):
  
  # Default direct options
  limit = fields['limit'] or 300
  obj_type = fields['object_type'] or 'asteroid'
  obj_state = fields['include']
  family = fields['family']
  orbit_uncertainty = fields['orbit_uncertainty']
  critical = fields["critical_list_numbered_object"]
  
  if obj_type == 'asteroid':
    asteroids = get_localdatabase()
    
    # Handle state
    if (None in obj_state) or ('provisional' in obj_state and 'numbered' in obj_state):
      bodies = asteroids
    elif 'provisional' in obj_state:
      bodies = asteroids[asteroids.is_provisional]
    elif 'numbered' in obj_state:
      bodies = asteroids[asteroids.is_numbered]

    bodies = bodies.drop(columns=['is_provisional', 'is_numbered'])
      
    # Handle family
    if None in family:
      bodies = asteroids
    elif 'tno' in family:
      bodies = asteroids[asteroids.is_tno]
    elif 'neo' in family:
      bodies = asteroids[asteroids.is_neo]
    elif 'mba' in family:
      bodies = asteroids[asteroids.is_mba]
    elif 'centaur' in family:
      bodies = asteroids[asteroids.is_centaur]
    elif 'jupiter_trojan' in family:
      bodies = asteroids[asteroids.is_jupiter_trojan]
      
    

# Function to standardize input data
def default(settings):
  
  # Default values
  if not settings['limit_magnitude']:
    settings['limit_magnitude'] = 16
  if not settings['exposition_time']:
    settings['exposition_time'] = 5
  if not settings['database']:
    settings['database'] = ["JPL", "MPC"]
  if not settings['observer']['code'] and not settings['observer']['coord']:
    settings['observer']['code'] = "geo"
  if not settings['epoch']:
    settings['epoch'] = {
      "range": {
        "start": str(Time.now()),
        "stop": str(Time.now() + 1 * u.day),
        "step": "1m",
        "number": 720
      }
    }
  if not settings['body']['id']:
    # settings['body'] = _search_bodies(settings['body']['fields'])
    args = _search_bodies(settings['body']['fields'])
    for i in args:
      print(f'{i}: {args[i]}')
    
    
  return settings

settings = get_config()
settings = default(settings)
# print(json.dumps(settings, indent=2))

object_type: asteroid
limit: 10
return_fields: name
